# Pashto FastSpeech2 on Colab

Runtime -> Change runtime type -> **GPU (T4 or better)**. Run the cells top to bottom.
Every step is a stage of `pipeline.py`; see README.md for what each one does.

## 1. Get the repo
Clones this repo. FastSpeech2 is bundled inside it (`FastSpeech2/`), so nothing else is cloned. For a private repo, put a token in the URL or make the repo public.

In [ ]:
REPO_URL = "https://github.com/msds25033/pashto-fastspeech2.git"
import os
if not os.path.exists("/content/pashto-fastspeech2"):
    !git clone -q {REPO_URL} /content/pashto-fastspeech2
%cd /content/pashto-fastspeech2
!git log -1 --format="%h %s"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install

In [ ]:
!pip install -q -r requirements.txt
!python pipeline.py setup          # bundled FastSpeech2: re-check patches, unzip vocoder, configs, lexicon
!python pipeline.py install-mfa    # Montreal Forced Aligner (~5 min)

## 3. (Optional) keep checkpoints and results on Google Drive
Recommended for long runs: a Colab session can end at any time. Training can then
continue in a new session with `--restore-step <last saved step>`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/pashto_tts/output /content/drive/MyDrive/pashto_tts/runs
!rm -rf FastSpeech2/output runs && ln -sfn /content/drive/MyDrive/pashto_tts/output FastSpeech2/output && ln -sfn /content/drive/MyDrive/pashto_tts/runs runs

## 4. Data: aligner corpus, alignment, features
`--clips` picks the clips; drop it to use all 759.

In [ ]:
CLIPS = '"1_crop-00[1-4]"'          # the 4 demo clips
!python pipeline.py corpus --clips {CLIPS}
!python pipeline.py align
!python pipeline.py preprocess

## 5. Train
Losses (train + validation), learning rate, sample spectrograms and audio are logged
to TensorBoard and exported to `runs/<run-name>/metrics/` when training ends.
Start TensorBoard first to watch it live.

**Already trained in an earlier session?** Skip the train cell. With step 3 (Drive)
and step 4 (data) run again, your checkpoints are back in `FastSpeech2/output/ckpt/Pashto`
and you can go straight to steps 6 and 7.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir FastSpeech2/output/log/Pashto

In [ ]:
!python pipeline.py train --run-name demo4 --steps 3000 --warmup 1000 --batch-size 4     --log-step 10 --val-step 100 --synth-step 500 --save-step 1000

### 5b. Keep training until it overfits (resume from the latest checkpoint)
Continues the same run from its newest checkpoint on Drive, on the same 4 clips. The
pipeline refuses to continue if `train.txt` holds different clips. `--overfit` switches
dropout off (encoder, decoder, and the duration/pitch/energy predictors), so the model can
memorise the clips instead of generalising. `--steps` is the new total.

In a new session: run steps 1–4 first, but not the training cell above.

In [ ]:
!python pipeline.py train --run-name demo4 --resume --overfit --steps 10000     --save-step 1000 --val-step 250 --synth-step 1000

## 6. Generate the two demo sentences (`data/demo_sentences.txt`)

In [ ]:
!python pipeline.py synthesize --run-name demo4

## 7. Generate from the training phones (overfitting test)
For each training clip: the phone sequence it was trained on, the original recording, and the
model's audio generated from that sequence alone, plus both spectrograms (original on top). `--step` picks the checkpoint (default: the latest).

In [ ]:
!python pipeline.py overfit-test --run-name demo4 --phones-only

In [ ]:
RUN = "demo4"
import glob, os
from IPython.display import Audio, Image, display, Markdown
test_dir = sorted(glob.glob(f"runs/{RUN}/overfit_test/step*"), key=lambda d: int(d.rsplit("step", 1)[1]))[-1]
print(test_dir)
for phones_file in sorted(glob.glob(f"{test_dir}/*_phones.txt")):
    clip = os.path.basename(phones_file)[:-len("_phones.txt")]
    display(Markdown(f"### {clip}"))
    print(open(phones_file, encoding="utf-8").read().strip())
    print("original:");          display(Audio(f"{test_dir}/{clip}_1_original.wav"))
    print("from the phones:");   display(Audio(f"{test_dir}/{clip}_5_from_phones.wav"))
    display(Image(f"{test_dir}/{clip}_mels.png"))   # top: original spectrogram, bottom: generated

## 8. Look, listen, download

In [ ]:
import glob
from IPython.display import Image, Audio, display
display(Image("runs/demo4/metrics/loss_curves.png"))
for wav in sorted(glob.glob("runs/demo4/audio/*.wav")):
    print(wav); display(Audio(wav))

In [ ]:
!cd runs && zip -qr /content/demo4_results.zip demo4
from google.colab import files
files.download("/content/demo4_results.zip")